In [1]:
import pandas as pd

In [2]:
sales = pd.read_csv("../data/raw/train.csv", low_memory=False)
stores = pd.read_csv("../data/raw/store.csv")

print("Sales dataset:", sales.shape)
print("Stores dataset:", stores.shape)

Sales dataset: (1017209, 9)
Stores dataset: (1115, 10)


In [3]:
sales.head()

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday
0,1,5,2015-07-31,5263,555,1,1,0,1
1,2,5,2015-07-31,6064,625,1,1,0,1
2,3,5,2015-07-31,8314,821,1,1,0,1
3,4,5,2015-07-31,13995,1498,1,1,0,1
4,5,5,2015-07-31,4822,559,1,1,0,1


In [4]:
print("SALES DATASET")
sales.info()

print("\nSTORE DATASET")
stores.info()

SALES DATASET
<class 'pandas.DataFrame'>
RangeIndex: 1017209 entries, 0 to 1017208
Data columns (total 9 columns):
 #   Column         Non-Null Count    Dtype
---  ------         --------------    -----
 0   Store          1017209 non-null  int64
 1   DayOfWeek      1017209 non-null  int64
 2   Date           1017209 non-null  str  
 3   Sales          1017209 non-null  int64
 4   Customers      1017209 non-null  int64
 5   Open           1017209 non-null  int64
 6   Promo          1017209 non-null  int64
 7   StateHoliday   1017209 non-null  str  
 8   SchoolHoliday  1017209 non-null  int64
dtypes: int64(7), str(2)
memory usage: 69.8 MB

STORE DATASET
<class 'pandas.DataFrame'>
RangeIndex: 1115 entries, 0 to 1114
Data columns (total 10 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Store                      1115 non-null   int64  
 1   StoreType                  1115 non-null   str    
 2   Assortment    

In [5]:
print("Missing values in sales:")
print(sales.isnull().sum())

print("\nMissing values in stores:")
print(stores.isnull().sum())

Missing values in sales:
Store            0
DayOfWeek        0
Date             0
Sales            0
Customers        0
Open             0
Promo            0
StateHoliday     0
SchoolHoliday    0
dtype: int64

Missing values in stores:
Store                          0
StoreType                      0
Assortment                     0
CompetitionDistance            3
CompetitionOpenSinceMonth    354
CompetitionOpenSinceYear     354
Promo2                         0
Promo2SinceWeek              544
Promo2SinceYear              544
PromoInterval                544
dtype: int64


In [6]:
print("Duplicate sales rows:", sales.duplicated().sum())
print("Duplicate store rows:", stores.duplicated().sum())

print("Duplicate Store-Date combinations:",
      sales.duplicated(subset=["Store", "Date"]).sum())

Duplicate sales rows: 0
Duplicate store rows: 0
Duplicate Store-Date combinations: 0


In [7]:
sales_clean = sales.copy()
stores_clean = stores.copy()

sales_clean["Date"] = pd.to_datetime(
    sales_clean["Date"],
    errors="raise"
)

print(sales_clean["Date"].dtype)
print("Earliest date:", sales_clean["Date"].min())
print("Latest date:", sales_clean["Date"].max())

datetime64[us]
Earliest date: 2013-01-01 00:00:00
Latest date: 2015-07-31 00:00:00


In [8]:
combined = sales_clean.merge(
    stores_clean,
    on="Store",
    how="left",
    validate="many_to_one"
)

print("Combined dataset shape:", combined.shape)
print("Stores without matching information:",
      combined["StoreType"].isna().sum())

combined.head()

Combined dataset shape: (1017209, 18)
Stores without matching information: 0


,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,5,2015-07-31,5263,555,1,1,0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
1,2,5,2015-07-31,6064,625,1,1,0,1,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,3,5,2015-07-31,8314,821,1,1,0,1,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct"
3,4,5,2015-07-31,13995,1498,1,1,0,1,c,c,620.0,9.0,2009.0,0,NaN,NaN,NaN
4,5,5,2015-07-31,4822,559,1,1,0,1,a,a,29910.0,4.0,2015.0,0,NaN,NaN,NaN


In [9]:
missing_summary = pd.DataFrame({
    "Missing Count": combined.isna().sum(),
    "Missing %": (combined.isna().mean() * 100).round(2)
})

missing_summary[missing_summary["Missing Count"] > 0]

,Missing Count,Missing %
CompetitionDistance,2642,0.26
CompetitionOpenSinceMonth,323348,31.79
CompetitionOpenSinceYear,323348,31.79
Promo2SinceWeek,508031,49.94
Promo2SinceYear,508031,49.94
PromoInterval,508031,49.94


In [10]:
print("Promo2 participation:")
print(stores_clean["Promo2"].value_counts())

print("\nMissing Promo2 start weeks:")
print(pd.crosstab(
    stores_clean["Promo2"],
    stores_clean["Promo2SinceWeek"].isna()
))

Promo2 participation:
Promo2
1    571
0    544
Name: count, dtype: int64

Missing Promo2 start weeks:
Promo2SinceWeek  False  True 
Promo2                       
0                    0    544
1                  571      0


In [11]:
print("Stores with missing competition distance:")
display(
    stores_clean.loc[
        stores_clean["CompetitionDistance"].isna(),
        ["Store", "CompetitionDistance",
         "CompetitionOpenSinceMonth",
         "CompetitionOpenSinceYear"]
    ]
)

print("\nMissing competition opening dates:")
print(stores_clean[
    ["CompetitionOpenSinceMonth", "CompetitionOpenSinceYear"]
].isna().sum())

Stores with missing competition distance:


,Store,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear
290,291,NaN,NaN,NaN
621,622,NaN,NaN,NaN
878,879,NaN,NaN,NaN



Missing competition opening dates:
CompetitionOpenSinceMonth    354
CompetitionOpenSinceYear     354
dtype: int64


In [12]:
cleaned = combined.copy()

# Remember which competition distances were missing
cleaned["CompetitionDistanceMissing"] = (
    cleaned["CompetitionDistance"].isna().astype(int)
)

# Fill missing distances using the median
median_distance = stores_clean["CompetitionDistance"].median()

cleaned["CompetitionDistance"] = (
    cleaned["CompetitionDistance"].fillna(median_distance)
)

print("Median competition distance:", median_distance)
print("Remaining missing competition distances:",
      cleaned["CompetitionDistance"].isna().sum())

print("Rows:", len(cleaned))

Median competition distance: 2325.0
Remaining missing competition distances: 0
Rows: 1017209


In [13]:
print("Negative sales:", (cleaned["Sales"] < 0).sum())
print("Zero sales:", (cleaned["Sales"] == 0).sum())

print("Closed stores:", (cleaned["Open"] == 0).sum())

print("Open stores with zero sales:",
      ((cleaned["Open"] == 1) &
       (cleaned["Sales"] == 0)).sum())

print("Negative customers:",
      (cleaned["Customers"] < 0).sum())

Negative sales: 0
Zero sales: 172871
Closed stores: 172817
Open stores with zero sales: 54
Negative customers: 0


In [14]:
open_zero_sales = cleaned[
    (cleaned["Open"] == 1) &
    (cleaned["Sales"] == 0)
]

print("Open stores with zero sales:", len(open_zero_sales))

display(
    open_zero_sales[
        ["Store", "Date", "Sales", "Customers", "Open", "Promo"]
    ].head(15)
)

print("\nCustomer counts in these records:")
print(open_zero_sales["Customers"].value_counts().head(10))

print("\nNumber of affected stores:")
print(open_zero_sales["Store"].nunique())

Open stores with zero sales: 54


,Store,Date,Sales,Customers,Open,Promo
86825,971,2015-05-15,0,0,1,0
142278,674,2015-03-26,0,0,1,0
196938,699,2015-02-05,0,0,1,1
322053,708,2014-10-01,0,0,1,1
330176,357,2014-09-22,0,0,1,0
340348,227,2014-09-11,0,0,1,0
340860,835,2014-09-11,0,0,1,0
341795,835,2014-09-10,0,0,1,0
346232,548,2014-09-05,0,0,1,1
346734,28,2014-09-04,0,0,1,1



Customer counts in these records:
Customers
0    52
3     1
5     1
Name: count, dtype: int64

Number of affected stores:
41


In [15]:
cleaned["Year"] = cleaned["Date"].dt.year
cleaned["Month"] = cleaned["Date"].dt.month
cleaned["Day"] = cleaned["Date"].dt.day
cleaned["WeekOfYear"] = cleaned["Date"].dt.isocalendar().week.astype(int)

display(
    cleaned[
        ["Store", "Date", "Sales", "Year", "Month", "Day", "WeekOfYear"]
    ].head()
)

,Store,Date,Sales,Year,Month,Day,WeekOfYear
0,1,2015-07-31,5263,2015,7,31,31
1,2,2015-07-31,6064,2015,7,31,31
2,3,2015-07-31,8314,2015,7,31,31
3,4,2015-07-31,13995,2015,7,31,31
4,5,2015-07-31,4822,2015,7,31,31


In [16]:
modeling_data = cleaned[
    (cleaned["Open"] == 1) &
    (cleaned["Sales"] > 0)
].copy()

modeling_data = modeling_data.sort_values(
    ["Date", "Store"]
).reset_index(drop=True)

print("Original records:", len(cleaned))
print("Modeling records:", len(modeling_data))
print("Excluded records:", len(cleaned) - len(modeling_data))

display(modeling_data.head())

Original records: 1017209
Modeling records: 844338
Excluded records: 172871


,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,...,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval,CompetitionDistanceMissing,Year,Month,Day,WeekOfYear
0,85,2,2013-01-01,4220,619,1,0,a,1,b,...,2011.0,0,NaN,NaN,NaN,0,2013,1,1,1
1,259,2,2013-01-01,6851,1444,1,0,a,1,b,...,NaN,0,NaN,NaN,NaN,0,2013,1,1,1
2,262,2,2013-01-01,17267,2875,1,0,a,1,b,...,2013.0,0,NaN,NaN,NaN,0,2013,1,1,1
3,274,2,2013-01-01,3102,729,1,0,a,1,b,...,NaN,1,10.0,2013.0,"Jan,Apr,Jul,Oct",0,2013,1,1,1
4,335,2,2013-01-01,2401,482,1,0,a,1,b,...,NaN,1,31.0,2013.0,"Jan,Apr,Jul,Oct",0,2013,1,1,1


In [17]:
from pathlib import Path

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

cleaned.to_csv(
    output_dir / "rossmann_cleaned.csv",
    index=False
)

modeling_data.to_csv(
    output_dir / "rossmann_modeling.csv",
    index=False
)

print("Datasets saved successfully!")

Datasets saved successfully!


In [18]:
sales.info()

<class 'pandas.DataFrame'>
RangeIndex: 1017209 entries, 0 to 1017208
Data columns (total 9 columns):
 #   Column         Non-Null Count    Dtype
---  ------         --------------    -----
 0   Store          1017209 non-null  int64
 1   DayOfWeek      1017209 non-null  int64
 2   Date           1017209 non-null  str  
 3   Sales          1017209 non-null  int64
 4   Customers      1017209 non-null  int64
 5   Open           1017209 non-null  int64
 6   Promo          1017209 non-null  int64
 7   StateHoliday   1017209 non-null  str  
 8   SchoolHoliday  1017209 non-null  int64
dtypes: int64(7), str(2)
memory usage: 69.8 MB


In [19]:
stores.info()

<class 'pandas.DataFrame'>
RangeIndex: 1115 entries, 0 to 1114
Data columns (total 10 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Store                      1115 non-null   int64  
 1   StoreType                  1115 non-null   str    
 2   Assortment                 1115 non-null   str    
 3   CompetitionDistance        1112 non-null   float64
 4   CompetitionOpenSinceMonth  761 non-null    float64
 5   CompetitionOpenSinceYear   761 non-null    float64
 6   Promo2                     1115 non-null   int64  
 7   Promo2SinceWeek            571 non-null    float64
 8   Promo2SinceYear            571 non-null    float64
 9   PromoInterval              571 non-null    str    
dtypes: float64(5), int64(2), str(3)
memory usage: 87.2 KB


In [20]:
print(sales.isnull().sum())


Store            0
DayOfWeek        0
Date             0
Sales            0
Customers        0
Open             0
Promo            0
StateHoliday     0
SchoolHoliday    0
dtype: int64


In [21]:
print(stores.isnull().sum())

Store                          0
StoreType                      0
Assortment                     0
CompetitionDistance            3
CompetitionOpenSinceMonth    354
CompetitionOpenSinceYear     354
Promo2                         0
Promo2SinceWeek              544
Promo2SinceYear              544
PromoInterval                544
dtype: int64


In [22]:
print(sales.duplicated().sum())

0


In [23]:
print(stores.duplicated().sum())

0


In [24]:
display(cleaned.head())
print(cleaned.shape)

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,...,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval,CompetitionDistanceMissing,Year,Month,Day,WeekOfYear
0,1,5,2015-07-31,5263,555,1,1,0,1,c,...,2008.0,0,NaN,NaN,NaN,0,2015,7,31,31
1,2,5,2015-07-31,6064,625,1,1,0,1,a,...,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct",0,2015,7,31,31
2,3,5,2015-07-31,8314,821,1,1,0,1,a,...,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct",0,2015,7,31,31
3,4,5,2015-07-31,13995,1498,1,1,0,1,c,...,2009.0,0,NaN,NaN,NaN,0,2015,7,31,31
4,5,5,2015-07-31,4822,559,1,1,0,1,a,...,2015.0,0,NaN,NaN,NaN,0,2015,7,31,31


(1017209, 23)


In [25]:
print(cleaned.columns.tolist())

['Store', 'DayOfWeek', 'Date', 'Sales', 'Customers', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday', 'StoreType', 'Assortment', 'CompetitionDistance', 'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear', 'Promo2', 'Promo2SinceWeek', 'Promo2SinceYear', 'PromoInterval', 'CompetitionDistanceMissing', 'Year', 'Month', 'Day', 'WeekOfYear']


In [26]:
import pandas as pd

synthetic = pd.DataFrame({
    "Store": [1] * 10,
    "Date": pd.date_range("2026-11-01", periods=10),
    "Sales": [1000, 1200, 1100, 1300, 900, 1400, 1000, 0, 1500, 1200],
    "Customers": [50, 60, 55, 65, 45, 70, 50, 0, 75, 60],
    "Open": [1, 1, 1, 1, 1, 1, 1, 0, 1, 1]
})

display(synthetic)
synthetic.to_csv("synthetic_sales.csv", index=False)

,Store,Date,Sales,Customers,Open
0,1,2026-11-01,1000,50,1
1,1,2026-11-02,1200,60,1
2,1,2026-11-03,1100,55,1
3,1,2026-11-04,1300,65,1
4,1,2026-11-05,900,45,1
5,1,2026-11-06,1400,70,1
6,1,2026-11-07,1000,50,1
7,1,2026-11-08,0,0,0
8,1,2026-11-09,1500,75,1
9,1,2026-11-10,1200,60,1


In [27]:
print(stores.isnull().sum())

Store                          0
StoreType                      0
Assortment                     0
CompetitionDistance            3
CompetitionOpenSinceMonth    354
CompetitionOpenSinceYear     354
Promo2                         0
Promo2SinceWeek              544
Promo2SinceYear              544
PromoInterval                544
dtype: int64
